In [4]:
import nltk
print("NLTK imported successfully")

NLTK imported successfully


In [1]:
# ==============================
# NLP Task 2: Text Classification
# ==============================

# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np
import re
import nltk
import os

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ==============================
# 2. DOWNLOAD NLTK DATA (if missing)
# ==============================
nltk_data_path = os.path.join(os.path.expanduser("~"), "nltk_data")
if not os.path.exists(nltk_data_path):
    os.makedirs(nltk_data_path)

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# ==============================
# 3. LOAD DATASET
# ==============================
df = pd.read_csv("dataset.zip")  # Make sure dataset.zip is in same folder

print("Columns:", df.columns)
print(df.head())

# ==============================
# 4. FIX COLUMN NAMES
# ==============================
df.columns = df.columns.str.strip().str.lower()
df.rename(columns={'review': 'text', 'sentiment': 'label'}, inplace=True)

print("\nAfter renaming:", df.columns)
print(df['label'].value_counts())

# ==============================
# 5. PREPROCESSING FUNCTION
# ==============================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text).lower()  # ensure string
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)
    
    # Remove punctuation/numbers
    text = re.sub(r"[^a-z\s]", "", text)
    
    # Tokenization
    try:
        tokens = word_tokenize(text)
    except LookupError:
        nltk.download('punkt', quiet=True)
        tokens = word_tokenize(text)
    
    # Remove stopwords & lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    
    return " ".join(tokens)

# Apply preprocessing
df['clean_text'] = df['text'].apply(preprocess_text)

print("\nSample cleaned text:")
print(df[['text', 'clean_text']].head())

# ==============================
# 6. FEATURE ENGINEERING
# ==============================
# Bag of Words
bow = CountVectorizer()
X_bow = bow.fit_transform(df['clean_text'])

# TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df['clean_text'])

# Labels
y = df['label']

# ==============================
# 7. TRAIN-TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# ==============================
# 8. MODEL TRAINING
# ==============================
# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

# Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# ==============================
# 9. EVALUATION FUNCTION
# ==============================
def evaluate_model(name, y_test, y_pred):
    print(f"\n{name} Results:")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

# Evaluate all models
evaluate_model("Logistic Regression", y_test, y_pred_lr)
evaluate_model("Naive Bayes", y_test, y_pred_nb)
evaluate_model("Decision Tree", y_test, y_pred_dt)

# ==============================
# 10. FINAL INSIGHTS
# ==============================
print("\nSummary:")
print("TF-IDF generally performs better than BoW.")
print("Logistic Regression works well for text classification.")
print("Naive Bayes is fast and effective for NLP tasks.")
print("Decision Tree may overfit on text data.")

Columns: Index(['review', 'sentiment'], dtype='str')
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

After renaming: Index(['text', 'label'], dtype='str')
label
positive    25000
negative    25000
Name: count, dtype: int64

Sample cleaned text:
                                                text  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                          clean_text  
0  one reviewer mentioned watchi